# 02. Feature Engineering
**Goal:** Transform raw data into a form models can learn from.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# Define columns
cols = ['unit', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]

train_df = pd.read_csv('../data/train_FD001.txt', sep='\s+', header=None, names=cols)
test_df = pd.read_csv('../data/test_FD001.txt', sep='\s+', header=None, names=cols)
rul_df = pd.read_csv('../data/RUL_FD001.txt', sep='\s+', header=None, names=['RUL'])

### Construct RUL Labels (Capped at 125)

In [2]:
# Calculate max cycle per engine
ru_max = train_df.groupby('unit')['cycle'].max().reset_index()
ru_max.columns = ['unit', 'max_cycle']

# Merge back and compute RUL
train_df = pd.merge(train_df, ru_max, on='unit')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']

# Apply Piecewise Linear RUL cap at 125
train_df['RUL'] = train_df['RUL'].clip(upper=125)

# Drop helper column
train_df.drop('max_cycle', axis=1, inplace=True)
train_df.head()

,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,125


### Drop Useless Sensors & Apply Rolling Features

In [3]:
# Drop flat sensors based on EDA
drop_sensors = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
train_df.drop(columns=drop_sensors, inplace=True)
test_df.drop(columns=drop_sensors, inplace=True)

# Remaining sensors
remaining_sensors = [col for col in train_df.columns if col.startswith('s')]

def apply_rolling_features(df, window=30):
    # Rolling mean and std grouped by unit (ignoring initial NaNs using min_periods=1)
    df_roll_mean = df.groupby('unit')[remaining_sensors].rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
    df_roll_std = df.groupby('unit')[remaining_sensors].rolling(window, min_periods=1).std().reset_index(level=0, drop=True)
    
    # Fill any remaining NaNs in std with 0 (for the first element in rolling window)
    df_roll_std = df_roll_std.fillna(0)
    
    # Rename columns
    df_roll_mean.columns = [f"{c}_avg_30" for c in df_roll_mean.columns]
    df_roll_std.columns = [f"{c}_std_30" for c in df_roll_std.columns]
    
    return pd.concat([df, df_roll_mean, df_roll_std], axis=1)

train_df = apply_rolling_features(train_df)
test_df = apply_rolling_features(test_df)

### Normalize Features (MinMaxScaler) & Save

In [4]:
# Separate features from targets/IDs
exclude_cols = ['unit', 'cycle', 'RUL', 'op1', 'op2', 'op3']  # Operations can be scaled or excluded depending on variance
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

scaler = MinMaxScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])



In [5]:
# Add binary classification label
train_df['will_fail_30'] = (train_df['RUL'] <= 30).astype(int)
# For test set, construct from RUL_FD001.txt
test_last_cycles = test_df.groupby('unit').last().reset_index()
rul_df_indexed = rul_df.copy()
rul_df_indexed['unit'] = range(1, len(rul_df)+1)
# Save this separately for classification tasks
rul_df_indexed['will_fail_30'] = (rul_df_indexed['RUL'] <= 30).astype(int)
rul_df_indexed.to_csv('../data/test_classification_labels.csv', index=False)


In [6]:
# Save processed datasets
train_df.to_csv('../data/processed_train.csv', index=False)
test_df.to_csv('../data/processed_test.csv', index=False)
print("Data saved successfully!")

Data saved successfully!
